In [ ]:
import torch
import transformers

from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, GPT2LMHeadModel, GPT2Tokenizer

from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

load_dotenv()

# === load teacher model ===
model_name = "gpt2"

teacher_model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # <- this should get rid of some warnings

teacher_model.eval()

# === prepare new model ===
student_model = transformers.models.GPT2LMHeadModel(teacher_model.config)


# === data generation ===
def synthetic_batch_generator(batch_size=1, seq_len=20, num_batches=1):
    for _ in range(num_batches):
        ids = teacher_model.generate(
            input_ids=None,
            attention_mask=None,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            max_new_tokens=seq_len - 1,
            num_return_sequences=batch_size,
        )

        # attention from 0th pos to first pad_token appearing after the 0th pos
        # the first appearing pad_token is included
        # it's a bit overly clever designed
        attention_mask = torch.zeros_like(ids, dtype=bool)
        attention_mask[:, 1:] = (
            ids[:, :-1] != tokenizer.pad_token_id
        )  # basically shifts ids one to the right
        attention_mask[:, :2] = 1

        yield ids, attention_mask


# === training loop ===

# training parameters
batch_size = 10
seq_len = 1024
num_batches = 15


# Move models to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher_model.to(device)
student_model.to(device)

# Optimizer
optimizer = AdamW(student_model.parameters(), lr=5e-5)

# Training loop
iteration = 0
for inputs, attention_mask in synthetic_batch_generator(
    batch_size=batch_size, seq_len=seq_len, num_batches=num_batches
):
    iteration += 1

    inputs = inputs.to(device)
    attention_mask = attention_mask.to(device)

    # Forward pass: student model
    outputs = student_model(inputs, attention_mask=attention_mask, labels=inputs)
    loss = outputs.loss

    # Backward pass
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"Epoch {iteration}, Loss: {loss.item()}")
    print(
        tokenizer.decode(
            student_model.generate(
                input_ids=None,
                attention_mask=None,
                pad_token_id=tokenizer.eos_token_id,
            )
        )
    )
    print(
        tokenizer.decode(
            student_model.generate(
                input_ids=None,
                attention_mask=None,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
            )
        )
    )

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch 1, Loss: 11.022467613220215
['<|endoftext|> penn<|endoftext|>']
['<|endoftext|>aucas Weirabsor PreviousIndividual Widees Uh Sina Appshovrite Technologies symbolic Lasocaust BoulderApplic LSbird']
Epoch 2, Loss: 8.688722610473633
['<|endoftext|><|endoftext|>']
['<|endoftext|> Netsrestlingrestling<|endoftext|>']
Epoch 3, Loss: 6.554801940917969
['<|endoftext|><|endoftext|>']
['<|endoftext|><|endoftext|>']
Epoch 4, Loss: 4.67595911026001
['<|endoftext|><|endoftext|>']
['<|endoftext|><|endoftext|>']
Epoch 5, Loss: 4.706308364868164
['<|endoftext|><|endoftext|>']
['<|endoftext|> put<|endoftext|>']
Epoch 6, Loss: 4.600361347198486
['<|endoftext|><|endoftext|>']
['<|endoftext|><|endoftext|>']
Epoch 7, Loss: 4.736991882324219
['<|endoftext|><|endoftext|>']
['<|endoftext|><|endoftext|>']
Epoch 8, Loss: 4.582930564880371
['<|endoftext|><|endoftext|>']
['<|endoftext|><|endoftext|>']
Epoch 9, Loss: 8.146514892578125
['<|endoftext|><|endoftext|>']
['<|endoftext|><|endoftext|>']
Epoch 10, Loss

In [106]:
for x, y in synthetic_batch_generator(batch_size=1, seq_len=20, num_batches=3):
    print(x)

tensor([[50256,   198,  1212,  2524,   318,  7256,   284,   262,   661,   286,
           262,  2059,   286, 10433,   290,   663,  5348,    13,   198,   198]])
tensor([[50256,    16,     8,   352,    11,   830, 23239,   583,  1218,   286,
         21408,   852, 25077,   656,   262,  4417,   286,   281,  7876,  4269]])
tensor([[50256,    25,   383,  5268,   286,   262,  5567,   198, 17811,  4603,
          1649,  3271, 28523,   274,  2627,  9677,   314,   335,   891, 26666]])


In [135]:
tokenizer.decode(student_model.generate(do_sample=True, temperature=3.0))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['<|endoftext|> 530Spirit lod thereto crest andenc slime:::: Nationwide 0 including attainedBuild<|endoftext|>']